# KneeXAI — GPU research runner for Kaggle

This notebook runs the repository's **existing** `scripts/run_research.py` on Kaggle GPU. It does not reimplement the metrics, so the resulting CSV follows the same validated contract used by Research Mode.

## Before running

1. In Kaggle Notebook settings, turn **Accelerator → GPU** on.
2. Attach a Kaggle Dataset containing the unpacked Research Toolkit source. Set `PROJECT_SOURCE` to the directory that contains `app/` and `scripts/`.
3. Attach a private Kaggle Model/Dataset containing one trusted `.pth` checkpoint per selected model ID. The notebook copies it under the name expected by `app.config` and exports matching ONNX in `/kaggle/working`.
4. For a full run, provide a final manifest with 50 verified test images per KL grade, plus the original 150-image manifest. Set both manifest paths below; the original must be staged as `inputs/test/test_manifest.json`.
5. Enable Internet only if Kaggle needs to install `quantus` and ONNX packages.

The shared `final_research` profile is defined in Python. Use `LIMIT = 1` only for a smoke check, with a run ID different from the full experiment. Model checkpoints, manifests, and image files are user-provided and are not included in the Toolkit.

In [ ]:
# Kaggle already provides CUDA-enabled PyTorch when GPU is enabled.
# Do NOT install torch from requirements.txt here: that file is intentionally CPU-oriented.
import subprocess
import sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'quantus==0.6.0', 'onnx==1.18.0', 'onnxruntime==1.20.1',
    'opencv-python-headless==4.10.0.84',
], check=True)

import onnx
import onnxruntime as ort
import torch
assert torch.cuda.is_available(), (
    'CUDA is unavailable. In Kaggle open Notebook options and select Accelerator → GPU, ' 
    'then restart the session and run this cell again.'
)
print('PyTorch:', torch.__version__)
print('ONNX:', onnx.__version__)
print('ONNX Runtime:', ort.__version__)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

# Set the Kaggle paths for the project and each selected model checkpoint.
PROJECT_SOURCE = Path('/kaggle/input/your-research-toolkit')
MODEL_SOURCE_DIRS = {
    'densenet121_class_aware': Path('/kaggle/input/your-densenet-checkpoint'),
    # Add the other model IDs and their own checkpoint folders before selecting 'all'.
}

# One model ID, or 'all' to run the four ordinal configurations sequentially.
MODEL_SELECTION = 'densenet121_class_aware'

# The final 250-image manifest must contain 50 labelled test images per KL grade.
# Relative image paths are resolved from the copied project root.
MANIFEST_SOURCE = Path('/kaggle/input/your-image-dataset/final_manifest_250.json')
ORIGINAL_MANIFEST_SOURCE = Path('/kaggle/input/your-image-dataset/test_manifest.json')

# The runner owns the one final profile: 50/1024 Faithfulness, 8 normalised MaxSensitivity samples.
EXPERIMENT_PROFILE = 'final_research'

# First run: False. To resume, keep this prefix and every setting below unchanged.
RESUME = False
RUN_ID_PREFIX = 'smoke-class-aware-v1'  # Use a new prefix for the full experiment.

LIMIT = 1  # Technical smoke test; use None only with a validated 250-image final manifest.
SEED = 42

assert PROJECT_SOURCE.is_dir(), f'Toolkit source directory not found: {PROJECT_SOURCE}'
assert (PROJECT_SOURCE / 'app/config.py').is_file(), f'Set PROJECT_SOURCE to the unpacked Toolkit root: {PROJECT_SOURCE}'
sys.path.insert(0, str(PROJECT_SOURCE))
from app.config import MODEL_CONFIGS, RESEARCH_MODEL_IDS
assert MODEL_SELECTION == 'all' or MODEL_SELECTION in RESEARCH_MODEL_IDS
SELECTED_MODELS = RESEARCH_MODEL_IDS if MODEL_SELECTION == 'all' else (MODEL_SELECTION,)
assert MANIFEST_SOURCE.is_file(), f'Manifest not found: {MANIFEST_SOURCE}'
assert EXPERIMENT_PROFILE == 'final_research'
MODEL_CHECKPOINTS = {}
for model_id in SELECTED_MODELS:
    source_dir = MODEL_SOURCE_DIRS.get(model_id)
    if source_dir is None or not source_dir.is_dir():
        raise FileNotFoundError(f'Set MODEL_SOURCE_DIRS[{model_id!r}] to its Kaggle checkpoint folder.')
    checkpoints = sorted(path for path in source_dir.glob('*.pth') if path.is_file())
    if len(checkpoints) != 1:
        raise ValueError(f'Expected exactly one .pth checkpoint in {source_dir}; found {[p.name for p in checkpoints]}')
    MODEL_CHECKPOINTS[model_id] = checkpoints[0]
print('Selected models:', SELECTED_MODELS)
print('Checkpoints:', MODEL_CHECKPOINTS)
print('Experiment profile:', EXPERIMENT_PROFILE)
print('Manifest:', MANIFEST_SOURCE)

### Manifest note

The runner validates every image before calculation. If your real manifest is in a separate Kaggle dataset, either use **absolute Kaggle paths** in `image_path`, or copy its images under the project directory and use paths relative to it. Do not create labels from directory names.

In [ ]:
import shutil
import subprocess

PROJECT_DIR = Path('/kaggle/working/knee-osteoarthritis-xai')
if PROJECT_DIR.exists():
    print(f'Reusing existing working copy: {PROJECT_DIR}. Remove or rename it manually only when starting a new experiment.')
else:
    shutil.copytree(PROJECT_SOURCE, PROJECT_DIR, ignore=shutil.ignore_patterns('.git', '.venv', '__pycache__'))

# A full final_research run verifies the original 150-image cohort at this exact path.
if LIMIT is None:
    if not ORIGINAL_MANIFEST_SOURCE.is_file():
        raise FileNotFoundError(f'Original 150-image manifest not found: {ORIGINAL_MANIFEST_SOURCE}')
    original_manifest_target = PROJECT_DIR / 'inputs/test/test_manifest.json'
    original_manifest_target.parent.mkdir(parents=True, exist_ok=True)
    if original_manifest_target.exists():
        if original_manifest_target.read_bytes() != ORIGINAL_MANIFEST_SOURCE.read_bytes():
            raise ValueError(f'Existing original manifest differs: {original_manifest_target}')
    else:
        shutil.copy2(ORIGINAL_MANIFEST_SOURCE, original_manifest_target)

models_dir = PROJECT_DIR / 'models'
models_dir.mkdir(exist_ok=True)
for model_id in SELECTED_MODELS:
    source = MODEL_CHECKPOINTS[model_id]
    checkpoint_target = models_dir / MODEL_CONFIGS[model_id]['checkpoint_path'].name
    shutil.copy2(source, checkpoint_target)
    print(f'{model_id}: {source.name} -> {checkpoint_target.name}')
    subprocess.run(
        [sys.executable, 'scripts/export_to_onnx.py', '--model-id', model_id],
        cwd=PROJECT_DIR, check=True,
    )

# A source manifest inside the project keeps its relative paths valid after copying.
# External absolute paths remain valid because Kaggle input datasets are mounted read-only.
if MANIFEST_SOURCE.is_relative_to(PROJECT_SOURCE):
    manifest_path = PROJECT_DIR / MANIFEST_SOURCE.relative_to(PROJECT_SOURCE)
else:
    manifest_path = MANIFEST_SOURCE

print('Project staged at:', PROJECT_DIR)
print('Results will be written under: /kaggle/working')

In [ ]:
# The repository runner owns the shared final profile and writes one CSV per model.
import subprocess

command = [
    sys.executable, 'scripts/run_research.py',
    '--manifest', str(manifest_path),
    '--profile', EXPERIMENT_PROFILE,
    '--output-dir', '/kaggle/working',
    '--seed', str(SEED),
    '--device', 'cuda',
]
if MODEL_SELECTION == 'all':
    command.append('--all-models')
    command += ['--run-id', RUN_ID_PREFIX]
    output_paths = [Path('/kaggle/working') / f'{RUN_ID_PREFIX}-{model_id}.csv' for model_id in SELECTED_MODELS]
else:
    selected_run_id = f'{RUN_ID_PREFIX}-{MODEL_SELECTION}'
    command += ['--model', MODEL_SELECTION, '--run-id', selected_run_id]
    output_paths = [Path('/kaggle/working') / f'{selected_run_id}.csv']
if LIMIT is not None:
    command += ['--limit', str(LIMIT)]
if RESUME:
    command.append('--resume')
print('Running:', ' '.join(command))
subprocess.run(command, cwd=PROJECT_DIR, check=True)

print('Completed:', *output_paths, sep='\n- ' )

In [ ]:
# Inspect and download the validated results.
import pandas as pd

for output_path in output_paths:
    results = pd.read_csv(output_path)
    print(f'\n{output_path.name}: {len(results)} rows')
    display(results.groupby(['metric_category', 'metric_name']).size().rename('rows').reset_index())
    display(results.groupby(['xai_method'])['metric_value'].agg(['count', 'median', 'mean']).reset_index())
    if 'attribution_l2_norm' in results:
        display(results.groupby('xai_method')['attribution_l2_norm'].agg(['median', 'min', 'max']).reset_index())

print('In the right Kaggle panel, use Output → Save Version or download this CSV and place it in results/research/ locally.')